# Fine-tune DARE3D — transfer learning (command-line twin)

Companion to `Run_dare3d_Prediction.ipynb` (runs a model) and `Run_dare3d_Retraining.ipynb`
(trains **from scratch**). This notebook **fine-tunes** a pretrained DARE3D checkpoint: it loads the
weights into the matching architecture, **freezes the backbone**, and continues training at a low
(optionally *discriminative*) learning rate — so you adapt an existing model to new data instead of
training from zero.

### One engine, three front-ends
Fine-tuning in DARE3D has a single **engine** — `dare3d/models/finetune.py` (`FineTuneMixin`) driven
by `experiment=finetune_{segmentation,regression}` + `model.finetune.*` Hydra overrides — and three
front-ends that all produce the *same* overrides:

| front-end | how you set the knobs |
|---|---|
| the napari **widget** (`napari_dare3d/_train_widget.py`) | GUI controls |
| **this notebook** | notebook variables |
| the **CLI** (`python dare3d/train.py experiment=finetune_… …`) | shell flags |

To make the equivalence *provable*, the run cells below call the **widget's own dispatch builder**,
`napari_dare3d._train.finetune_command(...)`, to produce the resolved override list — so what runs
here is byte-for-byte what the widget's *Start* button runs. DARE3D is **PyTorch-only**, Lightning +
Hydra, and fully 3D; there is no TensorFlow/`.h5`/backend-selector path.

### What this notebook verifies (two modes, §4 of the design)
- **In-process** — one tiny instrumented `trainer.fit` that reads *live* internals: the base weights
  landed in `self.net`, the freeze actually froze, the optimizer has the discriminative param-groups,
  the data has the right shapes, and — the decisive one — the **frozen BatchNorm3d survives a genuine
  mid-epoch validation boundary**.
- **Subprocess** — one integration run driving `dare3d/train.py` exactly as the widget does, then a
  **round-trip** reload through the *real* inference loader, and provenance-sidecar assertions.

Every check prints real numbers and a **PASS / FAIL / N/A(reason)** verdict; `N/A` is never laundered
into `PASS`.

In [ ]:
# --- Setup -----------------------------------------------------------------
# cuDNN gate: DARE3D 3D-convolution training segfaults on old cuDNN (8.x); set DARE3D_CUDNN=1 to keep
# cuDNN ON on a healthy stack (this env = dare3d-v2.0, torch 2.5 / cuDNN 9.1). See dare3d/train.py.
import os, sys, json, tempfile, warnings
from pathlib import Path
os.environ.setdefault("DARE3D_CUDNN", "1")

# Robust repo root: the folder holding dare3d/ + configs/ (walk up from cwd).
def _find_repo(start: Path) -> Path:
    for c in (start.resolve(), *start.resolve().parents):
        if (c / "dare3d").is_dir() and (c / "configs").is_dir():
            return c
    return start.resolve()
REPO = _find_repo(Path.cwd())
sys.path.insert(0, str(REPO))
os.environ.setdefault("PROJECT_ROOT", str(REPO))
CONFIGS = REPO / "configs"

import torch
from omegaconf import OmegaConf
try:
    OmegaConf.register_new_resolver("eval", eval, replace=True)   # dare3d configs use ${eval:...}
except Exception:
    pass
from napari_dare3d import _train                       # the widget's dispatch (napari-free)
from dare3d.models.finetune import load_net_state_dict, apply_freeze, FineTuneError

RESULTS = {}                                            # check-name -> "PASS"/"FAIL"/"N/A: reason"
def verdict(name, ok, extra=""):
    v = "PASS" if ok is True else ("FAIL" if ok is False else str(ok))
    RESULTS[name] = v
    print(f"[{v}] {name}" + (f"  ({extra})" if extra else ""))
    return ok

import lightning
print("repo        :", REPO)
print("torch       :", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| cuDNN:", torch.backends.cudnn.version())
print("lightning   :", lightning.__version__, "| DARE3D_CUDNN:", os.environ.get("DARE3D_CUDNN"))

## The three parameter categories (read this before touching any knob)

A checkpoint is not just weights — it is weights *of a specific shape*, trained on data of a specific
*geometry*, with a specific optimization recipe. When you fine-tune, some settings are **dictated by
the checkpoint** and some are **yours to choose**. Getting this wrong is the classic transfer-learning
failure: the model either refuses to load, or loads and silently transfers badly. DARE3D sorts every
fine-tune parameter into exactly one of three buckets.

**1. IMPOSED (architecture / weight-shape).** The pretrained tensors have fixed shapes, so the network
that receives them must be built identically: the net class, its channel widths and depth, the
segmentation head's output classes, the normalization type — and, *conditionally*, the input geometry
**iff the network has a size-dependent layer** (a flattened dense head, a fixed positional embedding, a
window-attention block). If you change any of these, `load_state_dict(strict=True)` raises a shape
mismatch. You do **not** pick these: this notebook **derives them from the base checkpoint** and shows
you the exact layer whose shape forces each one. In the widget, these are hidden/locked because a GUI
user can't see the base's shapes; here you *can*, so we make them visible and correct-by-construction
instead of throwing a dialog.

**2. RUNTIME-FREE (optimization + fully-convolutional geometry).** These do not change any weight
shape, so you set them freely: `ft_lr`, `backbone_lr_mult`, `weight_decay`, `lr_schedule`,
`warmup_epochs`, `grad_clip`, `patience`, `seed`, `augment`/`augment_strength`, `batch_size` — **plus**
the input geometry (`seg_crop_size`, `cell_radius`) *when the net is fully-convolutional*, because then
the conv weights are size-independent and any crop loads fine. "Free" does **not** mean "consequence-
free": a crop or radius far from the base's training value degrades transfer through a receptive-field
mismatch even though it loads; `bn_mode='adapt'` with a tiny `batch_size` re-estimates BatchNorm stats
from noisy mini-batches; and the *scratch* LR defaults are far too hot for weights that start pretrained
rather than random. This notebook prints the **base's training value next to each free knob** so any
divergence is visible *before* you run.

**3. FINE-TUNE-ONLY.** Meaningless when training from scratch: `freeze_preset` (what to freeze),
`unfreeze_last_stages` (how deep to thaw), and `bn_mode` (how to treat the frozen backbone's
BatchNorm). These are yours to choose and are explained where they appear.

The mapping table further down carries a **category** column matching the widget's classification
exactly.

## 1) Base model, stage, and data

`base_ckpt` is the pretrained checkpoint you adapt — a model-dir's `checkpoints/last.ckpt`. `stage` is
which network (`regression` = the division-axis head, `segmentation` = the centre-detection U-Net); one
checkpoint per stage. The dataset is the per-movie layout `<dataset>/<movie>/{im,label}/*.tif`; we
consolidate the chosen train/val movies into a split exactly as the widget's dispatch does.

We default to the **regression** stage on the published Gastruloid model: it is small and fast, and its
`RegressionNet` contains `BatchNorm3d`, so the decisive BN check below actually runs (segmentation works
identically — set `STAGE='segmentation'` and point `base_ckpt` at a seg model-dir).

In [ ]:
STAGE = "regression"                      # "regression" or "segmentation"
DATA = REPO / "DARE3d_data_190326" / "Gastruloid_241025"
BASE_DIR  = DATA / "weights" / (f"{STAGE}3d_exp10-b")
BASE_CKPT = BASE_DIR / "checkpoints" / "last.ckpt"
DATASET   = DATA / "trainingset"
TRAIN_MOVIES, VAL_MOVIES = ["movie3"], ["movie2"]   # disjoint specimens (val = held-out)

assert BASE_CKPT.is_file(), f"base checkpoint not found: {BASE_CKPT} (fetch the Zenodo data bundle)"
assert DATASET.is_dir(), f"dataset not found: {DATASET}"

# A private, disposable output dir for this notebook's runs (never touches the repo).
OUT = Path(tempfile.mkdtemp(prefix="dare3d_finetune_nb_"))
SPLIT = _train._resolve_split(str(DATASET), str(OUT), TRAIN_MOVIES, VAL_MOVIES)
print(f"stage      = {STAGE}")
print(f"base_ckpt  = {BASE_CKPT}")
print(f"dataset    = {DATASET}  (train={TRAIN_MOVIES}, val={VAL_MOVIES})")
print(f"split      = {SPLIT}")
print(f"out (temp) = {OUT}")

## 2) IMPOSED parameters — derived from the base, not chosen

The cell below opens the base checkpoint's saved `.hydra/config.yaml` and its `state_dict`, and reads
off the architecture the pretrained weights require. For each imposed value it prints **the specific
layer/shape that fixes it**. It also decides the geometry question empirically: it scans the
`state_dict` for any 2-D weight (a `nn.Linear` = a size-dependent dense head). If there is none, the
network is **fully convolutional** and the input crop is *runtime* (category 2); if there is one, the
crop is baked into that Linear's `in_features` and is therefore *imposed*.

These become **read-only** notebook variables (`IMPOSED[...]`). Do not edit them — the fine-tune
dispatch re-derives the exact same values from the base (`_train._net_overrides_from_base`) and passes
them as `model/net=… model.net.*` so the load matches by construction.

In [ ]:
cfg_base = OmegaConf.load(BASE_DIR / ".hydra" / "config.yaml")
sd = torch.load(BASE_CKPT, map_location="cpu", weights_only=False)["state_dict"]
net_sd = {k[len("net."):]: v for k, v in sd.items() if k.startswith("net.")}
sel = lambda k: OmegaConf.select(cfg_base, k)

net_class = str(sel("model.net._target_")).split(".")[-1]
linear_layers = [(k, tuple(v.shape)) for k, v in net_sd.items() if v.ndim == 2]   # nn.Linear
conv3d = sum(1 for v in net_sd.values() if v.ndim == 5)

IMPOSED = {"net_class": net_class,
           "start_filters": sel("model.net.start_filters"), "n_stages": sel("model.net.n_stages"),
           "channels": sel("model.net.channels"), "strides": sel("model.net.strides"),
           "out_channels": sel("model.net.out_channels"), "norm": sel("model.net.norm"),
           "base_crop_size": sel("crop_size"), "base_cell_radius": sel("cell_radius")}

print(f"IMPOSED architecture (from base {BASE_DIR.name}):")
print(f"  net_class      = {net_class!r}   ({conv3d} Conv3d weights, {len(linear_layers)} Linear weights)")
for k in ("start_filters", "n_stages", "channels", "strides", "out_channels", "norm"):
    if IMPOSED[k] is not None:
        print(f"  model.net.{k:<13}= {IMPOSED[k]}")

print("\nForcing layers (the shape that fixes each imposed value):")
if linear_layers:
    for name, shp in linear_layers:
        print(f"  SIZE-DEPENDENT HEAD  net.{name:<20} nn.Linear{shp}  -> in_features={shp[1]} is fixed by "
              f"start_filters*2^(n_stages-1) * (crop/2^n_stages)^3")
    print(f"  => geometry is IMPOSED for this net (crop is baked into the dense head). This repo fixes the "
          f"regression crop at {IMPOSED['base_crop_size']} and never exposes it, so there is no free knob.")
    GEOMETRY = "imposed"
else:
    prod = 1
    for s in (IMPOSED["strides"] or []):
        prod *= int(s[0] if isinstance(s, (list, tuple)) else s)
    print(f"  final Conv3d out_channels={IMPOSED['out_channels']} (segmentation head classes) — fixed by the last conv")
    print(f"  U-Net channels/strides/norm — fixed by every Conv3d/BatchNorm3d weight")
    print(f"  NO 2-D (Linear) weight anywhere -> the net is FULLY CONVOLUTIONAL -> the input crop is RUNTIME")
    print(f"  (only a HARD constraint remains: seg_crop_size must be divisible by prod(strides)={prod})")
    GEOMETRY = "runtime"
print(f"\nGEOMETRY verdict for {STAGE}: {GEOMETRY}")

## 3) RUNTIME-FREE parameters — yours to set (base value shown for each)

Editable knobs that do **not** change any weight shape. The cell prints the **base's training value**
beside each geometry knob so divergence is visible before the run, and states the interaction risks in
line. Guidance specific to fine-tuning (why the *scratch* defaults are wrong here):

- **`ft_lr` (default 1e-4).** Scratch LRs (≈1e-2–1e-1) are far too hot for pretrained weights — they
  erase the transferred features in the first few steps. Keep ≤1e-3.
- **`backbone_lr_mult` (0.1) + `discriminative`.** When you thaw part of the backbone, train it *slower*
  than the head (backbone LR = `ft_lr × backbone_lr_mult`); turning discriminative off makes them equal.
- **`batch_size` × `bn_mode`.** With `bn_mode='adapt'` a tiny batch re-estimates BatchNorm running stats
  from noisy mini-batches → prefer `'frozen'` or a larger batch.
- **geometry (`seg_crop_size`, `cell_radius`, segmentation only).** Fully-convolutional, so any value
  *loads*; but a crop/radius far from the base's training value degrades transfer (receptive-field
  mismatch). For **regression** these are inert — the crop is imposed-fixed (see cell 2).
- **`augment` / `augment_strength`.** `augment_strength` is the per-sample probability of applying the
  augmentation pipeline (wired to `data.augmentation.prob`); heavy augmentation on a fully-frozen
  backbone can hurt, since only the head can adapt to it.

In [ ]:
# --- runtime-free: EDIT THESE FREELY ---
ft_lr            = "1e-4"
backbone_lr_mult = "0.1"
discriminative   = True
weight_decay     = "1e-4"
lr_schedule      = "warmup_cosine"     # "warmup_cosine" (warmup->cosine) or "cosine"
warmup_epochs    = 2
grad_clip        = "1.0"               # gradient max-norm; "0" = off
patience         = 10                  # early-stop epochs w/o val improvement
seed             = 12345
augment          = True
augment_strength = "0.5"               # per-sample augmentation probability (0-1); 0 = none
batch_size       = 4
# geometry (segmentation only; inert for regression):
seg_crop_size    = 128
cell_radius      = 8

print("runtime-free knobs vs the base's training values:")
bc = IMPOSED["base_crop_size"]; br = IMPOSED["base_cell_radius"]
if STAGE == "segmentation":
    print(f"  seg_crop_size = {seg_crop_size:<5} | base trained at crop = {bc}"
          + ("   <-- DIVERGES: receptive-field/transfer risk" if bc is not None and seg_crop_size != bc else "   (matches base)"))
    print(f"  cell_radius   = {cell_radius:<5} | base trained at radius = {br}"
          + ("   <-- DIVERGES: target-scale shift" if br is not None and cell_radius != br else "   (matches base)"))
else:
    print(f"  (regression) seg_crop_size / cell_radius are inert; the regression crop is imposed at {bc}.")
print(f"  ft_lr = {ft_lr}  (fine-tune scale; <=1e-3 recommended, NOT the scratch LR)")
print(f"  batch_size = {batch_size}  x  bn_mode below  (adapt + tiny batch = noisy BN stats)")

## 4) FINE-TUNE-ONLY parameters — the transfer-learning recipe

These have no meaning for scratch training; they define *what* is frozen and *how*.

- **`freeze_preset`** — `encoder` (freeze the backbone, train the decoder/head), `encoder_partial`
  (also thaw the last `unfreeze_last_stages` stages), or `none` (full fine-tune). The engine maps this
  per net: for the U-Net it splits the monai encoder/decoder; for `RegressionNet` it freezes the
  residual `blocks` and trains the four heads.
- **`unfreeze_last_stages`** — with `encoder_partial`, how many of the deepest backbone stages to thaw.
- **`bn_mode`** — the frozen backbone's BatchNorm policy, and the subtlest correctness point in the
  whole feature (design §0):
  - `frozen` (default): frozen `BatchNorm3d` are put in `.eval()` so their running mean/var **and**
    affine stay fixed — a *true* freeze. This is enforced by overriding `LightningModule.train()`, not
    an epoch hook, because Lightning flips the module to `.eval()` for validation and back to
    `.train()` **mid-epoch** (when `val_check_interval < 1`); an epoch hook would miss that flip and let
    the "frozen" BN silently resume drifting. The in-process check below crosses exactly that boundary.
  - `adapt`: frozen BN stay in `.train()` so their running stats **re-estimate** on the new data (the
    affine weights remain frozen) — for a larger, distribution-shifted fine-tune set.

In [ ]:
# --- fine-tune-only: EDIT THESE FREELY ---
freeze_preset        = "encoder"        # encoder | encoder_partial | none
unfreeze_last_stages = 0                # >0 with encoder_partial
bn_mode              = "frozen"         # frozen (true freeze) | adapt (re-estimate running stats)

print(f"freeze_preset        = {freeze_preset}")
print(f"unfreeze_last_stages = {unfreeze_last_stages}")
print(f"bn_mode              = {bn_mode}")

## Mapping: notebook variable ↔ widget control ↔ Hydra override ↔ category

| notebook variable | widget control | Hydra override | category |
|---|---|---|---|
| `base_ckpt` | Regression/Segmentation base (.ckpt) | `model.finetune.base_ckpt` | selects the base |
| `STAGE` | Model (both/seg/reg) | `experiment=finetune_<stage>` | selects the stage |
| *(derived)* net class, channels, strides, out_channels, norm, start_filters, n_stages | — (hidden/locked) | `model/net=… model.net.*` | **imposed** |
| `freeze_preset` | Freeze preset | `model.finetune.freeze_preset` | **fine-tune-only** |
| `unfreeze_last_stages` | Unfreeze last N stages | `model.finetune.unfreeze_last_stages` | **fine-tune-only** |
| `bn_mode` | Frozen-backbone BN | `model.finetune.bn_mode` | **fine-tune-only** |
| `ft_lr` | Fine-tune LR | `model.finetune.ft_lr` | runtime-free |
| `discriminative` / `backbone_lr_mult` | Discriminative LR / Backbone LR × | `model.finetune.backbone_lr_mult` | runtime-free |
| `weight_decay` | Weight decay | `model.finetune.weight_decay` | runtime-free |
| `lr_schedule` / `warmup_epochs` | LR schedule / Warmup epochs | `model.finetune.lr_schedule` / `…warmup_epochs` | runtime-free |
| `grad_clip` | Grad clip | `model.finetune.grad_clip` → `trainer.gradient_clip_val` | runtime-free |
| `patience` | Early-stop patience | `model.finetune.patience` → `callbacks.early_stopping.patience` | runtime-free |
| `seed` | Seed | `model.finetune.seed` + top-level `seed` | runtime-free |
| `augment` / `augment_strength` | Augmentation / Augment strength | `data/augmentation=none` / `++data.augmentation.prob` | runtime-free |
| `batch_size` | (common) | `data.batch_size` | runtime-free |
| `seg_crop_size` / `cell_radius` | (common) | `crop_size` / `cell_radius` (seg only) | runtime (fully-conv) |

## 5) The resolved override list (diffable against the widget)

We assemble the free + fine-tune-only knobs into the `ft` dict and hand it to
`napari_dare3d._train.finetune_command(...)` — **the widget's own dispatch builder**. The printed list
is therefore exactly what the widget's *Start* button runs; the imposed `model/net=… model.net.*`
entries are the ones auto-derived from the base. `discriminative=False` collapses to
`backbone_lr_mult=1.0` (uniform LR).

In [ ]:
ft = dict(freeze_preset=freeze_preset, unfreeze_last_stages=int(unfreeze_last_stages), bn_mode=bn_mode,
          ft_lr=ft_lr, discriminative=bool(discriminative), backbone_lr_mult=backbone_lr_mult,
          weight_decay=weight_decay, lr_schedule=lr_schedule, warmup_epochs=int(warmup_epochs),
          grad_clip=grad_clip, augment=bool(augment), augment_strength=augment_strength,
          patience=int(patience), seed=int(seed), cell_radius=int(cell_radius),
          seg_crop_size=int(seg_crop_size))

# pre-flight (same as the widget): errors block, warnings would prompt. Shown for transparency.
pf = _train.finetune_preflight([STAGE], {STAGE: str(BASE_CKPT)}, ft)
print("pre-flight errors  :", pf["errors"] or "(none)")
print("pre-flight warnings:", pf["warnings"] or "(none)")

cmd = _train.finetune_command(STAGE, str(SPLIT), str(OUT), "demo", "demo", str(BASE_CKPT), ft,
                              epochs=50, batch_size=int(batch_size))
print("\nResolved Hydra overrides (identical to the widget's dispatch):")
for tok in cmd[2:]:
    print("  ", tok)

## 6) Verification — mode A: IN-PROCESS instrumented fit

One tiny real `trainer.fit` that reads live internals. We compose the config from the **same override
list** as above (plus tiny caps), instantiate the LightningModule, call `model.load_base()`, and fit
with `val_check_interval=0.5` so a validation runs **mid-epoch** — the boundary that would expose a
broken frozen-BN policy. To exercise the discriminative optimizer we thaw one stage
(`encoder_partial`, `unfreeze_last_stages=1`) for this diagnostic fit only.

Checks (each **PASS/FAIL/N/A**): (A) base weights loaded into `self.net`, key-complete and equal to the
base, differing from fresh init; (B) freeze deltas — frozen params unchanged vs base, trainable params
changed; (C) optimizer has the discriminative param-groups; (D) data shapes `(B,C,Z,Y,X)`, train/val
disjoint, inputs normalized; (E) the frozen `BatchNorm3d` stayed in `.eval()` and its running stats did
not move across the mid-epoch validation boundary (guarded — N/A if the net has no `BatchNorm3d`).

In [ ]:
import lightning as L
from lightning.pytorch import Callback
from lightning.pytorch.plugins.environments import SLURMEnvironment
SLURMEnvironment.detect = staticmethod(lambda: False)
from hydra import initialize_config_dir, compose
from hydra.utils import instantiate

# same dispatch overrides, but encoder_partial (to get 2 optimizer groups) + tiny caps
ft_fit = dict(ft, freeze_preset="encoder_partial", unfreeze_last_stages=1, warmup_epochs=1)
cmd_fit = _train.finetune_command(STAGE, str(SPLIT), str(OUT), "nbfit", "nbfit", str(BASE_CKPT),
                                  ft_fit, epochs=2, batch_size=2)
(OUT / "inproc").mkdir(parents=True, exist_ok=True)
# paths.output_dir is ${hydra:runtime.output_dir} in the config -> only resolves under hydra.main;
# under compose() we pin it (and work_dir) to a concrete temp path so Trainer can instantiate.
overrides = cmd_fit[2:] + ["+trainer.limit_train_batches=4", "+trainer.limit_val_batches=2",
                           "+trainer.val_check_interval=0.5", "data.num_workers=0",
                           "trainer.deterministic=warn", "+trainer.enable_progress_bar=false",
                           f"paths.output_dir={(OUT / 'inproc').as_posix()}",
                           f"paths.work_dir={OUT.as_posix()}"]
with initialize_config_dir(config_dir=str(CONFIGS), version_base="1.3"):
    cfg = compose(config_name="train", overrides=overrides)

model = instantiate(cfg.model)
fresh = {k: v.detach().cpu().clone() for k, v in model.net.state_dict().items()}   # random init
model.load_base()                                                                 # engine: base -> self.net
loaded = {k: v.detach().cpu().clone() for k, v in model.net.state_dict().items()}
base_net = {k[len("net."):]: v for k, v in
            torch.load(BASE_CKPT, map_location="cpu", weights_only=False)["state_dict"].items()
            if k.startswith("net.")}

# (A) weights loaded, key-complete, == base, != fresh
missing = [k for k in base_net if k not in loaded]
eq_base = all(torch.equal(loaded[k], base_net[k]) for k in base_net if k in loaded)
diff_fresh = sum(1 for k in fresh if not torch.equal(fresh[k], loaded[k]))
verdict("A weights loaded into self.net (key-complete, ==base, !=fresh)",
        (not missing) and eq_base and diff_fresh > 0,
        f"missing={len(missing)}, equal_to_base={eq_base}, changed_from_fresh={diff_fresh}/{len(fresh)}")

class Probe(Callback):
    def __init__(self): self.rec = []; self.bn = None; self.val_starts = 0
    def on_train_start(self, tr, pl):
        bns = [m for m in pl.net.modules() if isinstance(m, torch.nn.BatchNorm3d)
               and not any(p.requires_grad for p in m.parameters())]
        self.bn = bns[0] if bns else None
    def on_train_batch_start(self, tr, pl, batch, bi):
        if self.bn is not None:
            self.rec.append((pl.training, self.bn.training, self.bn.running_mean.detach().float().cpu().clone()))
    def on_validation_start(self, tr, pl): self.val_starts += 1

probe = Probe()
dm = instantiate(cfg.data)
trainer = instantiate(cfg.trainer, logger=False, enable_checkpointing=False, callbacks=[probe])
L.seed_everything(int(ft_fit["seed"]), workers=True)
trainer.fit(model, dm)

# (B) freeze deltas: frozen == base (unchanged), trainable != base (moved)
frozen_ok, trainable_moved = True, False
for n, p in model.net.named_parameters():
    if n not in base_net: continue
    same = torch.equal(p.detach().cpu(), base_net[n])
    if p.requires_grad:
        trainable_moved = trainable_moved or (not same)
    else:
        frozen_ok = frozen_ok and same
n_tr = sum(int(p.requires_grad) for p in model.net.parameters())
verdict("B freeze real (frozen unchanged vs base, trainable changed)", frozen_ok and trainable_moved,
        f"trainable_tensors={n_tr}, frozen_unchanged={frozen_ok}, trainable_moved={trainable_moved}")

# (C) optimizer param-groups + discriminative LR
try:
    opt = trainer.optimizers[0]
    base_lrs = [pg.get("initial_lr", pg["lr"]) for pg in opt.param_groups]
    ratio = (min(base_lrs) / max(base_lrs)) if max(base_lrs) else 0.0
    verdict("C optimizer has discriminative param-groups", len(opt.param_groups) == 2
            and abs(ratio - float(ft_fit["backbone_lr_mult"])) < 1e-6,
            f"groups={len(opt.param_groups)}, base_lrs={base_lrs}, ratio={ratio:.3g} (mult={ft_fit['backbone_lr_mult']})")
except Exception as e:
    verdict("C optimizer has discriminative param-groups", False, f"error: {e!r}")

# (D) data shapes / disjoint / normalization
try:
    xb = next(iter(dm.train_dataloader()))[0]["input"]
    tr_im = set(os.listdir(SPLIT / "train" / "im")); va_im = set(os.listdir(SPLIT / "val" / "im"))
    rng = (float(xb.min()), float(xb.max()))
    verdict("D data (B,C,Z,Y,X), train/val disjoint, normalized",
            xb.ndim == 5 and xb.shape[1] == 3 and tr_im.isdisjoint(va_im) and -0.01 <= rng[0] and rng[1] <= 1.01,
            f"shape={tuple(xb.shape)}, disjoint={tr_im.isdisjoint(va_im)}, range=({rng[0]:.3f},{rng[1]:.3f})")
except Exception as e:
    verdict("D data (B,C,Z,Y,X), train/val disjoint, normalized", False, f"error: {e!r}")

# (E) frozen BatchNorm3d across a genuine mid-epoch validation boundary
try:
    if probe.bn is None:
        verdict("E frozen-BN survives mid-epoch validation", "N/A: net has no frozen BatchNorm3d")
    else:
        eval_ok = all((not bn_tr) for (mod_tr, bn_tr, _) in probe.rec if mod_tr)
        rm0 = probe.rec[0][2]
        stats_frozen = all(torch.equal(rm0, r[2]) for r in probe.rec)
        crossed = probe.val_starts >= 2
        verdict("E frozen-BN eval + stats fixed across mid-epoch val", eval_ok and stats_frozen and crossed,
                f"records={len(probe.rec)}, always_eval_in_train={eval_ok}, stats_unchanged={stats_frozen}, "
                f"val_boundaries={probe.val_starts}")
except Exception as e:
    verdict("E frozen-BN survives mid-epoch validation", False, f"error: {e!r}")

## 6) Verification — mode B: SUBPROCESS integration + round-trip

Now the real thing: launch `dare3d/train.py` as a subprocess through the widget's streamer
(`_train._stream`, which sets `PYTHONUNBUFFERED`/UTF-8 and reads the pipe with `errors='replace'` — the
Windows-safe path, `deterministic='warn'` on cuDNN). It writes a model-dir with `checkpoints/last.ckpt`
+ a `finetune_config.json` provenance sidecar. We then **round-trip**: reload that checkpoint through
the *real* inference loader (`dare3d/predict.py::load_data`'s pattern — `instantiate(cfg.model)` +
`load_state_dict` + `net.to(device).eval()`) and run a forward, proving the fine-tuned model drops
straight into inference. Finally we assert the sidecar records the hyperparameters we set.

In [ ]:
cmd_sub = _train.finetune_command(STAGE, str(SPLIT), str(OUT), "nbsub", "nbsub", str(BASE_CKPT),
                                  ft, epochs=1, batch_size=2) + ["+trainer.limit_train_batches=3",
                                                                 "+trainer.limit_val_batches=2"]
print("subprocess:", " ".join(cmd_sub[2:5]), "...\n")
try:
    n = 0
    for line in _train._stream(cmd_sub, None):
        n += 1
        if n <= 2 or n % 60 == 0: print("  |", line[:105])
    stream_ok = True
except Exception as e:
    stream_ok = False; print("  subprocess error:", repr(e))
model_dir = _train.model_dirs(str(OUT), "nbsub", "nbsub")[STAGE]
ckpt = model_dir / "checkpoints" / "last.ckpt"
sidecar = model_dir / "checkpoints" / "finetune_config.json"
verdict("B1 subprocess produced ckpt + sidecar", stream_ok and ckpt.is_file() and sidecar.is_file(),
        f"stream_ok={stream_ok}, ckpt={ckpt.is_file()}, sidecar={sidecar.is_file()}")

# round-trip through the REAL inference loader (predict.py::load_data pattern)
try:
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    cfg_ft = OmegaConf.load(model_dir / ".hydra" / "config.yaml")
    m2 = instantiate(cfg_ft.model)
    m2.load_state_dict(torch.load(ckpt, map_location="cpu", weights_only=False)["state_dict"])
    m2.net = m2.net.to(dev); m2.net.eval()
    crop = int(sel("crop_size")) if STAGE == "regression" else 64
    with torch.no_grad():
        out = m2.net(torch.randn(1, 3, crop, crop, crop, device=dev))
    ok_fwd = isinstance(out, dict) and len(out) > 0
    verdict("B2 round-trip: fine-tuned ckpt reloads via the inference loader + forward runs", ok_fwd,
            f"output keys={list(out.keys()) if isinstance(out, dict) else type(out)}")
except Exception as e:
    verdict("B2 round-trip: fine-tuned ckpt reloads via the inference loader + forward runs", False, f"error: {e!r}")

# provenance sidecar reflects the GUI/notebook values
try:
    info = json.loads(sidecar.read_text()); hp = info["hyperparameters"]
    sc_ok = (hp["bn_mode"] == ft["bn_mode"] and hp["freeze_preset"] == ft["freeze_preset"]
             and abs(float(hp["ft_lr"]) - float(ft["ft_lr"])) < 1e-9
             and info["base_model"]["sha256"] and info["environment"]["torch"])
    verdict("B3 sidecar records the run's provenance + hyperparameters", bool(sc_ok),
            f"bn_mode={hp['bn_mode']}, freeze_preset={hp['freeze_preset']}, ft_lr={hp['ft_lr']}, "
            f"base_sha256={str(info['base_model']['sha256'])[:12]}, torch={info['environment']['torch']}")
except Exception as e:
    verdict("B3 sidecar records the run's provenance + hyperparameters", False, f"error: {e!r}")

## 6) Verification — mode C: the IMPOSED constraints actually bind

Proof that the values the notebook *derived* from the base (cell 2) are the ones the weights require:
we rebuild the network with a **wrong** imposed value (`start_filters`) and show `load_state_dict`
rejects it, while the derived value loads clean and its head shape matches. This is the same guarantee
the widget relies on — the engine's strict, stage-aware load — shown from the notebook side.

In [ ]:
if STAGE == "regression":
    from dare3d.models.components.simple_regression_net import RegressionNet
    good = RegressionNet(input_channels=[-1, 0, 1], im_size=int(IMPOSED["base_crop_size"]),
                         n_stages=int(IMPOSED["n_stages"]), start_filters=int(IMPOSED["start_filters"]))
    load_net_state_dict(good, str(BASE_CKPT), stage=STAGE)                    # inherited -> clean
    head_in = good.head1_len[0].in_features
    wrong = RegressionNet(input_channels=[-1, 0, 1], im_size=int(IMPOSED["base_crop_size"]),
                          n_stages=int(IMPOSED["n_stages"]), start_filters=int(IMPOSED["start_filters"]) // 2)
    try:
        load_net_state_dict(wrong, str(BASE_CKPT), stage=STAGE); raised = False
    except FineTuneError:
        raised = True
    verdict("C imposed constraints bind (derived loads clean; wrong start_filters rejected)",
            raised and head_in == linear_layers[0][1][1],
            f"derived head in_features={head_in} (==Linear shape {linear_layers[0][1]}); "
            f"wrong start_filters rejected={raised}")
else:
    # segmentation: the derived channels/strides load; a wrong channel width is rejected
    from dare3d.models.components.multiscale_unet import MultiScaleUNet
    ch = list(IMPOSED["channels"]); st = list(IMPOSED["strides"])
    good = MultiScaleUNet(spatial_dims=3, in_channels=3, out_channels=int(IMPOSED["out_channels"]),
                          channels=ch, strides=st, norm=IMPOSED["norm"], num_res_units=3, bias=False,
                          dropout=0.0, output_names=[], downsample_factors=[1])
    load_net_state_dict(good, str(BASE_CKPT), stage=STAGE)
    wrong = MultiScaleUNet(spatial_dims=3, in_channels=3, out_channels=int(IMPOSED["out_channels"]),
                           channels=[c // 2 for c in ch], strides=st, norm=IMPOSED["norm"],
                           num_res_units=3, bias=False, dropout=0.0, output_names=[], downsample_factors=[1])
    try:
        load_net_state_dict(wrong, str(BASE_CKPT), stage=STAGE); raised = False
    except FineTuneError:
        raised = True
    verdict("C imposed constraints bind (derived channels load; halved channels rejected)", raised,
            f"wrong channels rejected={raised}")

## 7) Summary & using your fine-tuned model

Below is the consolidated PASS/FAIL/N/A table for every check. To use a fine-tuned model, point
inference at its model-dir (the folder holding `.hydra/config.yaml` + `checkpoints/`) exactly as you
would a released one:

- **napari** — *DARE3D inference* widget, set the Segmentation/Regression model-dir field.
- **notebook** — `Run_dare3d_Prediction.ipynb`, set the model dir.
- **CLI** — `python dare3d/predict.py +segmentation.model_dir=… +regression.model_dir=… +inference_dir=…`.

Because the fine-tuned checkpoint is byte-compatible with the inference loader (round-trip B2), there is
nothing else to convert.

In [ ]:
print("=== DARE3D fine-tune notebook — verification summary ===")
for k, v in RESULTS.items():
    print(f"  {v:<6} {k}")
n_fail = sum(1 for v in RESULTS.values() if v == "FAIL")
print(f"\n{sum(1 for v in RESULTS.values() if v=='PASS')} PASS / "
      f"{n_fail} FAIL / {sum(1 for v in RESULTS.values() if v.startswith('N/A'))} N/A")
print("ALL GOOD" if n_fail == 0 else "SOME CHECKS FAILED — see above")